# Implementing the GPT Training Loop

We have a model architecture and a way to prepare our data, but how do we actually *teach* the model? The training loop is the heart of this process; it's the conductor that orchestrates the flow of data, tells the model when to learn, and tracks its progress. By the end of this notebook, you will understand the fundamental mechanics of a training loop and be able to implement one from scratch to train a simple GPT model.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

# Set a seed for reproducibility
torch.manual_seed(42)

### The Core Idea: The Teacher and the Student

Imagine you're teaching a student to write poetry. You can't just give them a library of books and expect them to learn. Instead, you create a structured lesson plan. This is our training loop.

1.  **Give a Prompt (Get a Batch):** You give the student a starting line, "The sun rises over the..." This is like getting a batch of data.
2.  **Student Writes (Forward Pass):** The student tries to complete the line, perhaps writing "...hill." This is the model making a prediction.
3.  **Grade the Attempt (Calculate Loss):** You compare the student's attempt to a correct example (e.g., "...calm sea."). You calculate a "grade" or "error score" based on how different they are. This is the loss function.
4.  **Explain the Mistake (Backward Pass):** You don't just say "wrong." You explain *why* it was wrong, noting which parts of their thinking led to the error. This is backpropagation, which calculates how much each part of the model (each "neuron") contributed to the final mistake.
5.  **Student Learns (Update Weights):** The student adjusts their mental model based on your feedback. They learn that "sun rises" is often associated with "sea" in the poems they've seen. This is the optimizer step, where the model's internal parameters (weights) are updated.

The training loop simply repeats this process thousands of times, with thousands of different prompts, until the student (our model) becomes a proficient poet.

In [ ]:
# ---- 1. Mock Components ----

# A mock data loader that yields batches of token IDs
def get_batch(split, block_size, batch_size):
    # Let's create some dummy data. In reality, this comes from a file.
    # We'll create a simple sequence 0, 1, 2, ..., 99
    data = torch.arange(100)
    # Let's say the first 80 tokens are for training, the rest for validation
    split_idx = 80
    train_data, val_data = data[:split_idx], data[split_idx:]
    
    current_data = train_data if split == 'train' else val_data
    
    # Get random starting points for our batches
    ix = torch.randint(len(current_data) - block_size, (batch_size,))
    
    # Create the input batch (x) and target batch (y)
    x = torch.stack([current_data[i:i+block_size] for i in ix])
    y = torch.stack([current_data[i+1:i+block_size+1] for i in ix])
    return x, y

# A very simple mock GPT model
class MockGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        # Each token directly maps to an embedding
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # A simple linear layer for prediction
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # Get token embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        # Get logits
        logits = self.lm_head(tok_emb) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Reshape for cross_entropy
            B, T, C = logits.shape
            logits_for_loss = logits.view(B*T, C)
            targets_for_loss = targets.view(B*T)
            loss = F.cross_entropy(logits_for_loss, targets_for_loss)
            
        return logits, loss
    
    def parameters(self):
        return super().parameters()


# ---- 2. The Core Training Loop ----

def run_training():
    # Hyperparameters
    max_iters = 1000
    learning_rate = 1e-3
    vocab_size = 100 # Our dummy data has 100 unique "tokens"
    block_size = 8
    batch_size = 4
    n_embd = 16

    # Model and Optimizer initialization
    model = MockGPT(vocab_size, block_size, n_embd)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    print("Starting training...")
    # The training loop
    for step in range(max_iters):
        # 1. Get a batch of data
        xb, yb = get_batch('train', block_size, batch_size)

        # 2. Forward pass: get predictions and loss
        logits, loss = model(xb, yb)
        
        # 3. Backward pass: compute gradients
        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        # 4. Update weights
        optimizer.step()
        
        if step % 100 == 0:
            print(f"Step {step:4d}, Loss: {loss.item():.4f}")
            
    print("Training finished.")
    return model

# We will run this function in the demonstration cell below.
# run_training()

### Walking Through the Code

Let's break down the `run_training` function line by line.

**1. Mock Components:**
*   `get_batch(...)`: This function simulates our data pipeline. It grabs a random chunk of our training or validation data and returns an input tensor `x` and a target tensor `y`. In a real scenario, this would read from a pre-processed file of token IDs.
*   `MockGPT`: This is a stripped-down version of our full GPT model. It has an embedding layer to turn token IDs into vectors and a single linear layer (`lm_head`) to predict the next token. Crucially, its `forward` method calculates the `cross_entropy` loss, which measures how surprised the model was by the correct answer.

**2. The `run_training` Function:**
*   **Hyperparameters**: We define key constants like `max_iters` (how many learning steps to take) and `learning_rate` (how large of a step the optimizer should take).
*   **Model and Optimizer**: We create an instance of our `MockGPT` and an `AdamW` optimizer. The optimizer is a sophisticated algorithm for updating the model's weights; we simply tell it which parameters it's responsible for (`model.parameters()`).
*   **The `for` loop**: This is the main engine. For each `step`:
    *   `xb, yb = get_batch(...)`: We fetch our "prompt" (`xb`, the input) and the "correct answer" (`yb`, the target).
    *   `logits, loss = model(xb, yb)`: The model makes its prediction (`logits`) and calculates its own error score (`loss`).
    *   `optimizer.zero_grad()`: We erase any leftover calculations from the previous step. It's like wiping a blackboard clean before solving a new problem.
    *   `loss.backward()`: This is where PyTorch works its magic. It traces the loss backward through every operation in the model to calculate the gradient for each parameter—essentially determining how much each weight contributed to the error.
    *   `optimizer.step()`: The optimizer uses the gradients calculated in the previous step to update all the model's weights, nudging them in the right direction to reduce the error on the next attempt.
*   **Logging**: The `if step % 100 == 0:` block is just for us to see what's happening. It prints the current loss every 100 steps, which should hopefully be decreasing over time!

In [ ]:
# --- Demonstration: Running the Training Loop ---
# This cell executes the core logic we defined above.
# Notice how the loss steadily decreases as the model learns.

trained_model = run_training()

### Going Deeper: Estimating Loss Periodically

In the `nanoGPT` `train.py` script, you won't see `print(loss.item())` inside the main loop. Why not? The loss from a single batch can be very noisy and jump around wildly. It's like judging a student's entire progress based on one pop quiz—it's not a reliable measure.

A much better approach is to pause training periodically and estimate the loss over several batches, then average the results. This gives a smoother, more statistically meaningful view of the model's performance. It also allows us to check performance on the validation set—data the model hasn't trained on—to see if it's truly learning or just memorizing the training data (overfitting).

Let's implement this more robust evaluation strategy. We'll create a new function, `estimate_loss`, that the main training loop will call every so often.

In [ ]:
# --- Visualization: Plotting Training and Validation Loss ---
# Let's write a more advanced training loop that tracks loss over time
# and plots the results.

@torch.no_grad() # Tell PyTorch we aren't calculating gradients here
def estimate_loss(model, eval_iters, block_size, batch_size):
    """Estimates loss over a few batches for both train and val splits."""
    out = {}
    model.eval() # Set model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, block_size, batch_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Set model back to training mode
    return out

def run_training_with_eval():
    # Hyperparameters
    max_iters = 1000
    learning_rate = 1e-3
    eval_interval = 100 # How often to evaluate
    eval_iters = 50     # How many batches to average for evaluation
    vocab_size = 100
    block_size = 8
    batch_size = 4
    n_embd = 16

    # Model and Optimizer
    model = MockGPT(vocab_size, block_size, n_embd)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # For plotting
    iterations = []
    train_losses = []
    val_losses = []

    print("Starting training with evaluation...")
    for step in range(max_iters):
        # Periodically evaluate and store losses
        if step % eval_interval == 0:
            losses = estimate_loss(model, eval_iters, block_size, batch_size)
            iterations.append(step)
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])
            print(f"Step {step:4d}: Train Loss {losses['train']:.4f}, Val Loss {losses['val']:.4f}")

        # Perform a regular training step
        xb, yb = get_batch('train', block_size, batch_size)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
    print("Training finished.")
    
    # Plotting the results
    plt.figure(figsize=(10, 5))
    plt.plot(iterations, train_losses, label='Training Loss')
    plt.plot(iterations, val_losses, label='Validation Loss')
    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss Over Time')
    plt.legend()
    plt.grid(True)
    plt.show()

run_training_with_eval()

### Summary

**What we built:** We constructed a complete, runnable training loop in PyTorch. We started with a simple loop that demonstrated the core concepts and then evolved it into a more robust version that periodically evaluates performance on both training and validation data, producing a learning curve visualization.

**Key Takeaways:**
*   The training loop is a cycle: get data, perform a forward pass, calculate loss, perform a backward pass, and update weights.
*   The `optimizer` handles the complex task of updating model weights based on the gradients computed during the backward pass.
*   Evaluating loss on a single batch is noisy. Averaging the loss over many batches (`eval_iters`) gives a much more stable and reliable picture of the model's performance.
*   Plotting validation loss alongside training loss is crucial for identifying overfitting, where the model performs well on data it has seen but fails to generalize to new, unseen data.

This loop is the engine that drives the learning in any deep learning project, and the patterns you learned here are fundamental to training not just GPT models, but nearly any neural network.